# Football IQ — Player Performance Dashboard



In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

pd.set_option('display.max_columns', 60)
sns.set_style('whitegrid')

PROCESSED = "data/processed/"


## 1. Load data

In [4]:
players = pd.read_csv(PROCESSED + "clean_players_with_value.csv")
clusters = pd.read_csv(PROCESSED + "player_clusters.csv")

CLUSTER_NAMES = {
    0: "Centre-back",
    1: "Ball-winner / Presser",
    2: "Out-and-out Striker",
    3: "Attacking Mid / Forward",
    4: "Ball-playing Defender",
}
clusters['cluster_name'] = clusters['cluster'].map(CLUSTER_NAMES)

print(players.shape, clusters.shape)
players.head(2)


(24239, 68) (833, 17)


,index,Player,#,Nation,Pos,Age,Min,Gls,Ast,PK,PKatt,Sh,SoT,CrdY,CrdR,Touches,Tkl,Int,Blocks,xG,npxG,xAG,SCA,GCA,Cmp_x,Att_x,Cmp%_x,PrgP,Carries,PrgC,...,Att_y,Cmp%_y,Att_y.1,Thr,Launch%,AvgLen,Att_y.2,Launch%.1,AvgLen.1,Opp,Stp,Stp%,#OPA,AvgDist,home,game_id,season,team,opponent,Gls_p90,Ast_p90,Sh_p90,SoT_p90,Tkl_p90,Int_p90,Touches_p90,PrgP_p90,PrgC_p90,player_name_norm,market_value_in_eur
0,0,Ivan Toney,17.0,eng ENG,FW,25-150,90,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,33.0,1.0,0.0,0.0,0.0,0.0,0.1,1.0,0.0,11.0,26.0,42.3,4.0,11.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,2021-2022,Brentford,Arsenal,0.0,0.0,0.000000,0.0,1.000000,0.0,33.000000,4.000000,0.000000,ivan toney,NaN
1,1,Bryan Mbeumo,19.0,cm CMR,FW,22-006,85,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,35.0,1.0,0.0,1.0,0.3,0.3,0.1,3.0,0.0,15.0,23.0,65.2,1.0,17.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,2021-2022,Brentford,Arsenal,0.0,0.0,2.117647,0.0,1.058824,0.0,37.058824,1.058824,2.117647,bryan mbeumo,NaN


## 2.Season summary 

In [6]:
p90_cols = ['Gls_p90','Ast_p90','Sh_p90','SoT_p90','Tkl_p90','Int_p90','Touches_p90','PrgP_p90','PrgC_p90']

profile = (players.groupby(['Player','season','Pos','team'])
           .agg(matches_played=('game_id','nunique'),
                total_minutes=('Min','sum'),
                total_goals=('Gls','sum'),
                total_assists=('Ast','sum'),
                avg_market_value=('market_value_in_eur','mean'),
                **{c: (c,'mean') for c in p90_cols})
           .reset_index())

profile = profile.merge(clusters[['Player','season','cluster','cluster_name']], on=['Player','season'], how='left')
print(f"{profile['Player'].nunique()} unique players across {profile['season'].nunique()} seasons")
profile.head()


862 unique players across 3 seasons


,Player,season,Pos,team,matches_played,total_minutes,total_goals,total_assists,avg_market_value,Gls_p90,Ast_p90,Sh_p90,SoT_p90,Tkl_p90,Int_p90,Touches_p90,PrgP_p90,PrgC_p90,cluster,cluster_name
0,Aaron Connolly,2021-2022,FW,Brighton,3,141,0.0,0.0,NaN,0.000000,0.000000,1.500000,0.000000,0.000000,0.000000,18.000000,0.833333,1.833333,NaN,NaN
1,Aaron Connolly,2021-2022,LM,Brighton,1,15,0.0,0.0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,30.000000,0.000000,0.000000,NaN,NaN
2,Aaron Cresswell,2021-2022,CB,West Ham,2,180,0.0,0.0,500000.0,0.000000,0.000000,0.000000,0.000000,0.500000,1.500000,78.500000,2.500000,1.000000,4.0,Ball-playing Defender
3,Aaron Cresswell,2021-2022,"CB,LB",West Ham,2,180,0.0,0.0,500000.0,0.000000,0.000000,0.500000,0.000000,1.000000,1.500000,83.000000,2.000000,0.500000,4.0,Ball-playing Defender
4,Aaron Cresswell,2021-2022,LB,West Ham,26,2276,2.0,2.0,500000.0,0.076923,0.076923,0.673077,0.115385,0.884615,1.119505,75.901099,7.076923,2.171703,4.0,Ball-playing Defender


## 3. League averages 

In [8]:
league_avg = profile[p90_cols].mean()
league_avg


Gls_p90         0.176157
Ast_p90         0.118194
Sh_p90          1.642617
SoT_p90         0.561402
Tkl_p90         2.094416
Int_p90         0.920748
Touches_p90    62.127792
PrgP_p90        4.103844
PrgC_p90        2.536202
dtype: float64

## 4. Interactive player profile 

In [ ]:
all_players = sorted(profile['Player'].unique())

def render_profile(player_name):
    rows = profile[profile['Player']==player_name].sort_values('season')
    if len(rows) == 0:
        print("No data for this player.")
        return

    latest = rows.iloc[-1]
    display(Markdown(f"## {player_name}"))
    display(Markdown(
        f"**Position:** {latest['Pos']}  |  **Latest team:** {latest['team']}  |  "
        f"**Playing style cluster:** {latest['cluster_name'] if pd.notna(latest['cluster_name']) else 'n/a (low minutes)'}"
    ))
    if pd.notna(latest['avg_market_value']) and latest['avg_market_value'] > 0:
        display(Markdown(f"**Estimated market value:** €{latest['avg_market_value']:,.0f}"))
    else:
        display(Markdown("**Estimated market value:** not available in transfer dataset"))

    display(rows[['season','team','matches_played','total_minutes','total_goals','total_assists']]
            .rename(columns={'season':'Season','team':'Team','matches_played':'Matches',
                              'total_minutes':'Minutes','total_goals':'Goals','total_assists':'Assists'})
            .set_index('Season'))

    fig, axes = plt.subplots(1, 2, figsize=(13,4))

    if len(rows) > 1:
        axes[0].plot(rows['season'], rows['total_goals'], marker='o', label='Goals')
        axes[0].plot(rows['season'], rows['total_assists'], marker='o', label='Assists')
        axes[0].set_title('Goals & assists by season')
        axes[0].legend()
    else:
        axes[0].bar(['Goals','Assists'], [latest['total_goals'], latest['total_assists']], color=['#2e7d32','#1976d2'])
        axes[0].set_title(f"{latest['season']} totals")

    vals = latest[p90_cols].astype(float)
    comp = pd.DataFrame({'Player': vals.values, 'League avg': league_avg.values}, index=p90_cols)
    comp.plot(kind='barh', ax=axes[1], color=['#c62828','#9e9e9e'])
    axes[1].set_title(f"Per-90 profile vs league average ({latest['season']})")

    plt.tight_layout()
    plt.show()

def prompt_player():
    while True:
        name = input("Enter player name (e.g. Mohamed Salah): ").strip()
        match = next((p for p in all_players if p.lower() == name.lower()), None)
        if match:
            return match
        partial = [p for p in all_players if name.lower() in p.lower()]
        if len(partial) == 1:
            return partial[0]
        elif len(partial) > 1:
            print(f'  Multiple matches: {", ".join(partial[:8])}{"..." if len(partial)>8 else ""}. Be more specific.')
        else:
            print(f'  No player found matching "{name}". Try just their surname, or check spelling.')

PLAYER_NAME = prompt_player()
render_profile(PLAYER_NAME)


Enter player name (e.g. Mohamed Salah):  Son Hung man


  No player found matching "Son Hung man". Try just their surname, or check spelling.


Enter player name (e.g. Mohamed Salah):  Alexandra trend-Anorld 


  No player found matching "Alexandra trend-Anorld". Try just their surname, or check spelling.


## 5. Squad view 



In [12]:
team_dd = widgets.Dropdown(options=sorted(profile['team'].dropna().unique()), description='Team:')
season_dd = widgets.Dropdown(options=sorted(profile['season'].unique()), description='Season:')
squad_out = widgets.Output()

def render_squad(*args):
    with squad_out:
        clear_output()
        sub = profile[(profile['team']==team_dd.value) & (profile['season']==season_dd.value)]
        sub = sub.sort_values('total_minutes', ascending=False)
        display(sub[['Player','Pos','cluster_name','matches_played','total_minutes','total_goals','total_assists']]
                .rename(columns={'cluster_name':'Style'}))

team_dd.observe(render_squad, names='value')
season_dd.observe(render_squad, names='value')
display(widgets.HBox([team_dd, season_dd]), squad_out)
render_squad()


Output()